In [ ]:
import xarray as xr
import metpy.calc as mpcalc
from metpy.units import units
import os

In [ ]:
# Step 1: Open the U and V fields from CESM output



''' Data Source Settings '''

# 3d->2d varaibles
lev_var = 200.
vars_2d = ['U','V']

#print(gc.__version__)

# LENS2

in_case = 'b.e21.BHISTcmip6.f09_g17.LE2-1001.001'
in_case_suff = ['.185001-185912','.186001-186912','.187001-187912','.188001-188912','.189001-189912']



## LENS1

#in_case = 'b.e11.B1850C5CN.f09_g16.005'
#in_case_suff = ['.040001-049912']




var_div = 'DIV'+str(int(lev_var))
var_u = 'U'+str(int(lev_var))
var_v = 'V'+str(int(lev_var))





nc_dir = '/glade/work/rneale/python-netcdf/enso/'

in_dir_u = nc_dir+var_u+'/'
in_dir_v = nc_dir+var_v+'/'
out_dir = nc_dir+var_div+'/'

for itime,ic_suff in enumerate(in_case_suff):

        
    
        ic_suff = ic_suff+'.nc'
    
        '''  None Run Source Specific Settings '''
        
        in_case_pref = in_case+'.cam.h0.'
        
        
        
        
    
        out_file = in_case_pref+var_div+ic_suff
        

        in_ufile = in_case_pref+var_u+ic_suff
        in_vfile = in_case_pref+var_v+ic_suff
               
        os.makedirs(out_dir, exist_ok=True)
        
        
        
        # Extract the data needed
        
        print('- Read in files...')
        print(in_dir_u+in_ufile)
        print(in_dir_v+in_vfile)
        
        
        ds_u = xr.open_mfdataset(in_dir_u+in_ufile,parallel=True)
        ds_v = xr.open_mfdataset(in_dir_v+in_vfile,parallel=True)

    

        u = ds_u[var_u].metpy.quantify()
        v = ds_v[var_v].metpy.quantify()
      
        lat = ds_u['lat'].metpy.quantify()
        lon = ds_u['lon'].metpy.quantify()

        # Calculate grid spacing (meters)
        dx, dy = mpcalc.lat_lon_grid_deltas(lon, lat)

        # Compute divergence
        print('- Calculating Divergence')
        da_div = mpcalc.divergence(u, v, dx=dx, dy=dy)
        da_div = div.metpy.dequantify()
        da_div.name = 'divergence'

    
        
        
#        div = geocat.comp.divergence(u, v, lat=ds_u['lat'], lon=ds_u['lon'])



        # Step 3: Add metadata
        div.name = "divergence"
        div.attrs["long_name"] = "Horizontal divergence"
        div.attrs["units"] = "1/s"

# Step 4: Save to NetCDF
#        div.to_dataset().to_netcdf()
    
        print('- Writing out...')
        da_var.to_netcdf(out_dir+out_file,mode="w")
        
        print('-Done')












